# Lab 5: Regression und Klassifikation

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Datenordner finden: Notebook liegt in labs/ oder loesungen/, die Daten in data/
DATA = next(p for p in [Path("data"), Path("../data"), Path("../../data")] if p.exists())
print("Datenordner:", DATA)

Dieses Lab gehört zu **Teil 5: Überwachtes Lernen: Regression und Klassifikation**.

## Lernziele

- Den Gradientenabstieg für eine Gerade in NumPy nachbauen und die Wirkung der Lernrate beobachten
- Eine lineare Regression trainieren, ihre Koeffizienten lesen und nach dem Skalieren vergleichen
- Daten in der richtigen Reihenfolge vorbereiten: erst teilen, dann skalieren
- Mit `predict_proba` eine eigene Schwelle setzen und die Folgen in der Konfusionsmatrix ablesen
- Fünf Klassifikationsverfahren in einer Schleife trainieren und in einer Tabelle nebeneinanderstellen

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, recall_score,
                             confusion_matrix, classification_report)

## Block 1: Gradientenabstieg von Hand

Die Übungsdaten sind 100 Punkte um die Gerade y = 2.5 · x + 1.0, mit etwas Rauschen. Der Gradientenabstieg soll die Steigung `m` und den Achsenabschnitt `b` aus den Punkten zurückgewinnen.

1. Übernehmen Sie die Schleife von der Folie: Start bei `m = 0`, `b = 0`, Lernrate `alpha = 0.01`, 1000 Schritte. Erwartet: `m` = 2.508, `b` = 0.944.
2. Zählen Sie, nach wie vielen Schritten der MSE unter 1.1 fällt: für `alpha` 0.001 und 0.01. Probieren Sie danach `alpha = 0.05`. Erwartet: 61 Schritte und 4 Schritte. Bei 0.05 explodieren die Werte, am Ende steht `nan`.
3. Sammeln Sie den MSE je Schritt in einer Liste (`alpha = 0.001`, 200 Schritte), zeichnen Sie die Kurve und vergleichen Sie Ihr Ergebnis aus Aufgabe 1 mit `LinearRegression`. Erwartet: `LinearRegression` liefert `m` = 2.508 und `b` = 0.949.

In [ ]:
# Übungsdaten für Block 1 (wahre Werte: m = 2.5, b = 1.0)
rng = np.random.default_rng(42)
x_g = rng.uniform(0, 10, 100)
y_g = 2.5 * x_g + 1.0 + rng.normal(0, 1, 100)

fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(x_g, y_g, s=10)
ax.set_xlabel("x (Merkmal)")
ax.set_ylabel("y (Zielgröße)")
ax.set_title("100 Punkte um eine Gerade")
plt.show()

In [ ]:
# Aufgabe 1: Gradientenabstieg für eine Gerade
# Tipp: dm = -2 * np.mean(x_g * (y_g - y_hat)), db = -2 * np.mean(y_g - y_hat)
m, b, alpha = 0.0, 0.0, 0.01
for schritt in range(1000):
    y_hat = m * x_g + b
    dm = ...          # Ableitung des MSE nach m
    db = ...          # Ableitung des MSE nach b
    # m = m - alpha * dm
    # b = b - alpha * db
print(m, b)

In [ ]:
# Aufgabe 2: Lernrate variieren und Schritte zählen
# Tipp: die Schleife in eine Funktion packen, nach jedem Schritt den MSE berechnen
def schritte_bis(alpha, grenze=1.1, max_schritte=1000):
    m, b = 0.0, 0.0
    for schritt in range(1, max_schritte + 1):
        ...           # Schritt wie in Aufgabe 1, danach: mse = np.mean((y_g - (m * x_g + b)) ** 2)
        # if mse < grenze:
        #     return schritt, mse
    return None

for alpha in [0.001, 0.01, 0.05]:
    print(alpha, schritte_bis(alpha))

In [ ]:
# Aufgabe 3: MSE-Kurve zeichnen und mit LinearRegression vergleichen
# Tipp: verlauf.append(mse) in der Schleife; LinearRegression().fit(x_g.reshape(-1, 1), y_g)
verlauf = []
# ... Schleife mit alpha = 0.001 und 200 Schritten

# fig, ax = plt.subplots()
# ax.plot(verlauf)

# lr = LinearRegression().fit(x_g.reshape(-1, 1), y_g)
# print(lr.coef_, lr.intercept_)

## Block 2: Lineare Regression auf California Housing

Jede Zeile von `california_housing.csv` ist ein Bezirk in Kalifornien. Zielgröße ist `MedHouseVal`, der mittlere Hauswert in 100.000 USD. `MedInc` ist das mittlere Einkommen in 10.000 USD.

1. Teilen Sie die Daten (`test_size=0.2`, `random_state=1`), trainieren Sie `LinearRegression` und geben Sie MSE und R² auf den Testdaten aus. Erwartet: 16512 Trainings- und 4128 Testzeilen, MSE 0.529, R² 0.597.
2. Geben Sie die Koeffizienten als sortierte `Series` aus. Welcher ist der stärkste positive, welcher der stärkste negative? Erwartet: `AveBedrms` 0.632 und `Longitude` -0.441, Achsenabschnitt -37.52.
3. Sagen Sie den Hauswert für den vorgegebenen Bezirk vorher. Erhöhen Sie danach `MedInc` um 1 und sagen Sie erneut vorher. Erwartet: 2.37 (rund 237.000 USD). Der Unterschied ist 0.439, genau der Koeffizient von `MedInc`.
4. Skalieren Sie die Merkmale mit `StandardScaler` (`fit` nur auf den Trainingsdaten), trainieren Sie die lineare Regression neu und sortieren Sie die Koeffizienten nach Betrag. Erwartet: `Latitude` -0.910, `Longitude` -0.885, `MedInc` 0.830 liegen vorn, `Population` fast null (-0.004). R² bleibt 0.597: Skalieren ändert die Vorhersage nicht, macht aber die Koeffizienten vergleichbar.

In [ ]:
haeuser = pd.read_csv(DATA / "california_housing.csv")
print(haeuser.shape)
X_h = haeuser.drop(columns="MedHouseVal")
y_h = haeuser["MedHouseVal"]
haeuser.head(3)

In [ ]:
# Aufgabe 1: Split, fit, predict, Kennzahlen
# Tipp: dasselbe Muster wie beim ersten Modell, nur mit LinearRegression
X_h_train, X_h_test, y_h_train, y_h_test = ..., ..., ..., ...
linreg = ...
# y_h_pred = linreg.predict(X_h_test)
# print(mean_squared_error(y_h_test, y_h_pred), r2_score(y_h_test, y_h_pred))

In [ ]:
# Aufgabe 2: Koeffizienten lesen
# Tipp: pd.Series(linreg.coef_, index=X_h.columns).sort_values()
koef = ...
koef

In [ ]:
# Aufgabe 3: Vorhersage für einen Bezirk
# Tipp: predict erwartet eine Tabelle mit denselben Spalten wie beim Training
bezirk = pd.DataFrame([{"MedInc": 5.0, "HouseAge": 20, "AveRooms": 6.0, "AveBedrms": 1.0,
                        "Population": 1000, "AveOccup": 3.0, "Latitude": 34.0, "Longitude": -118.0}])
wert = ...
# reicher = bezirk.assign(MedInc=6.0)
# ...

In [ ]:
# Aufgabe 4: Koeffizienten nach dem Skalieren vergleichen
# Tipp: scaler.fit_transform(X_h_train), scaler.transform(X_h_test), danach LinearRegression wie in Aufgabe 1
scaler_h = StandardScaler()
linreg_s = ...
# koef_s = pd.Series(linreg_s.coef_, index=X_h.columns)
# ...

## Block 3: Logistische Regression auf dem Brustkrebs-Datensatz

569 Gewebeproben mit je 30 Messwerten. Achtung bei der Kodierung der Zielgröße: **0 = bösartig (malignant), 1 = gutartig (benign)**. Prüfen Sie das immer mit `target_names`.

1. Prüfen Sie die Kodierung (`target_names`, `np.bincount`). Teilen Sie dann die Daten (`test_size=0.2`, `random_state=1`) und skalieren Sie danach mit `StandardScaler`. Erwartet: 212 bösartig, 357 gutartig, 455 Trainings- und 114 Testzeilen. Mittelwert der ersten skalierten Spalte: 0.0 im Training, -0.119 im Test.
2. Verständnisfrage: Was ist falsch an `X_s = scaler.fit_transform(X)` mit anschließendem `train_test_split(X_s, y)`? Schreiben Sie Ihre Antwort als Text in die Variable `antwort`.
3. Trainieren Sie `LogisticRegression(max_iter=1000)` auf den skalierten Daten und geben Sie Accuracy, Konfusionsmatrix und `classification_report` aus. Erwartet: Accuracy 0.974, Matrix `[[40 2] [1 71]]`, also 2 bösartige Proben übersehen.
4. Holen Sie mit `predict_proba` die Wahrscheinlichkeit für „gutartig" und vergleichen Sie die Schwelle 0.5 mit 0.3. Bauen Sie danach die Tabelle für 0.3, 0.5, 0.7, 0.9. Erwartet: bösartig übersehen 3, 2, 2, 1. Gutartig fälschlich als bösartig 0, 1, 1, 7.

In [ ]:
data = load_breast_cancer()            # im Paket enthalten, kein Netz nötig
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
print(X.shape)
X[["mean area", "mean smoothness"]].describe().T[["mean", "std"]].round(3)   # sehr verschiedene Skalen

In [ ]:
# Aufgabe 1: Kodierung prüfen, teilen, dann skalieren
# Tipp: fit_transform nur auf den Trainingsdaten, auf den Testdaten nur transform
print(data.target_names)
# print(np.bincount(y))
X_train, X_test, y_train, y_test = ..., ..., ..., ...
scaler = StandardScaler()
X_train_s = ...
X_test_s = ...
# print(X_train_s[:, 0].mean().round(3), X_test_s[:, 0].mean().round(3))

In [ ]:
# Aufgabe 2: Verständnisfrage zur Falle "Skalieren vor dem Split"
#   X_s = scaler.fit_transform(X)
#   X_train, X_test, y_train, y_test = train_test_split(X_s, y, ...)
antwort = "..."
print(antwort)

In [ ]:
# Aufgabe 3: Logistische Regression trainieren und bewerten
# Tipp: confusion_matrix(y_test, y_pred): Zeilen = Wahrheit, Spalten = Vorhersage, Klasse 0 zuerst
logreg = ...
# y_pred = logreg.predict(X_test_s)
# print(accuracy_score(y_test, y_pred))
# print(confusion_matrix(y_test, y_pred))
# print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
# Aufgabe 4: Wahrscheinlichkeiten und eigene Schwelle
# Tipp: Spalte 1 von predict_proba ist P(Klasse 1) = P(gutartig); (proba >= 0.3).astype(int)
proba = ...
# for schwelle in [0.5, 0.3]:
#     y_schwelle = ...
#     print(schwelle)
#     print(confusion_matrix(y_test, y_schwelle))

## Block 4: Entscheidungsbaum und Baumtiefe

Bäume vergleichen immer nur ein Merkmal mit einer Schwelle. Sie brauchen keine Skalierung, deshalb arbeiten Sie hier mit `X_train` und `X_test` in Originaleinheiten.

1. Trainieren Sie `DecisionTreeClassifier(max_depth=2, random_state=1)` und geben Sie den Baum mit `export_text` aus. Erwartet: erste Frage `worst perimeter <= 106.05`, Accuracy auf den Testdaten 0.886.
2. Trainieren Sie Bäume mit `max_depth` 1, 2, 3, 5 und `None`. Tabellieren Sie Trainings- und Testgenauigkeit. Erwartet: Training 0.930, 0.958, 0.969, 1.000, 1.000. Test 0.877, 0.886, 0.912, 0.947, 0.947. Der unbegrenzte Baum wird 5 Ebenen tief.

In [ ]:
# Aufgabe 1: Ein Baum zum Vorlesen
# Tipp: export_text(baum, feature_names=list(X.columns))
baum = ...
# baum.fit(X_train, y_train)
# print(export_text(baum, feature_names=list(X.columns)))
# print(baum.score(X_test, y_test))

In [ ]:
# Aufgabe 2: Baumtiefe gegen Genauigkeit
# Tipp: Liste von Dictionaries, am Ende pd.DataFrame; baum.get_depth() liefert die erreichte Tiefe
zeilen = []
for tiefe in [1, 2, 3, 5, None]:
    ...
pd.DataFrame(zeilen)

## Block 5: Modellvergleich in einer Schleife

Alle Modelle in scikit-learn haben dieselbe Schnittstelle. Fünf Verfahren passen deshalb in eine Schleife. Verwenden Sie die skalierten Daten `X_train_s` und `X_test_s` (den Bäumen schadet das nicht).

1. Trainieren Sie logistische Regression, Entscheidungsbaum (`max_depth=3`), Random Forest (100 Bäume), SVM (RBF) und kNN (k = 5) in einer Schleife und geben Sie eine Tabelle mit Trainings- und Testgenauigkeit aus. Erwartet, Test: 0.974, 0.912, 0.956, 0.974, 0.956.
2. Ergänzen Sie je Modell die Zahl der übersehenen bösartigen Proben und den Recall für die Klasse bösartig (`recall_score(..., pos_label=0)`). Erwartet, übersehen: 2, 6, 5, 2, 5. Recall: 0.952, 0.857, 0.881, 0.952, 0.881.
3. Verständnisfrage: Zwischen dem besten und dem schwächsten Modell liegen 0.062 Accuracy. Wie viele Testfälle sind das? Schreiben Sie die Rechnung in die Zelle. Erwartet: 7 von 114.

In [ ]:
# Aufgabe 1: Fünf Modelle, eine Schleife
modelle = {
    "Logistische Regression": LogisticRegression(max_iter=1000),
    "Entscheidungsbaum": DecisionTreeClassifier(max_depth=3, random_state=1),
    # drei weitere Modelle ergänzen: Random Forest, SVM (RBF), kNN (k=5)
}
zeilen = []
for name, modell in modelle.items():
    ...
pd.DataFrame(zeilen)

In [ ]:
# Aufgabe 2: Übersehene bösartige Proben und Recall für bösartig
# Tipp: confusion_matrix(...)[0, 1] = wirklich 0 (bösartig), vorhergesagt 1 (gutartig)
zeilen = []
for name, modell in modelle.items():
    ...
pd.DataFrame(zeilen)

In [ ]:
# Aufgabe 3: Wie viele Testfälle sind 0.062 Accuracy?
testfaelle = 114          # so viele Zeilen hat der Testteil, siehe len(y_test)
unterschied = ...
print(unterschied)

## Zusatzaufgaben

1. Wie weit kommt man mit wenigen Merkmalen? Trainieren Sie `LinearRegression` nur mit `MedInc`, danach mit `MedInc`, `Latitude` und `Longitude`, und tabellieren Sie das R² auf den Testdaten neben dem Modell mit allen acht Merkmalen. Erwartet: R² 0.472, 0.579 und 0.597.
2. Zeichnen Sie ein Streudiagramm `y_h_test` gegen `y_h_pred` (lineare Regression aus Block 2). Wo sehen Sie die Deckelung der Zielgröße? Erwartet: senkrechte Punktreihe bei 5.0 (188 Testbezirke), das Modell sagt dort im Mittel nur 4.02 vorher (kleinster Wert 0.90, größter 7.22).
3. Ändern Sie `random_state` im Split auf 0 und 2 und wiederholen Sie den Modellvergleich. Erwartet: mit 0 liegt die SVM vorn (0.982), mit 2 teilen sich logistische Regression und kNN den ersten Platz (0.974), der Baum bleibt bei 0.912.

In [ ]:
# Zusatz 1: R² mit einem, drei und allen acht Merkmalen
zeilen = []
for spalten in [["MedInc"], ["MedInc", "Latitude", "Longitude"], list(X_h.columns)]:
    ...
pd.DataFrame(zeilen)

In [ ]:
# Zusatz 2: Wahrheit gegen Vorhersage
# fig, ax = plt.subplots()
# ax.scatter(..., ..., s=3, alpha=0.3)

In [ ]:
# Zusatz 3: Anderer Split, andere Reihenfolge?
# Tipp: nach jedem neuen Split auch neu skalieren
for rs in [0, 2]:
    ...

## Was Sie mitnehmen

- Training heißt: einen Verlust minimieren. Der Gradientenabstieg geht schrittweise bergab, die Lernrate entscheidet über zu langsam, passend oder Absturz.
- Koeffizienten sagen, wie stark und in welche Richtung ein Merkmal wirkt. Vergleichen lassen sie sich erst nach dem Skalieren.
- Erst teilen, dann skalieren. Prüfen Sie die Kodierung der Zielgröße (`target_names`), und lesen Sie neben der Accuracy immer die Konfusionsmatrix: Die Schwelle ist eine fachliche Entscheidung.